# Zambia VACS 2014 — Covariate Codebook (Step 2)

Compiles the **covariate codebook** for Zambia 2014 using the standard 5-column format:
**Category**, **Variable**, **Type**, **Format**, **Questions**.

**Data sources:**
- `.dta` PUDs: `ZAMBIA_VACS_2014_Male_PUD.dta` / `Female_PUD.dta` (variable names + labels)
- Codebook PDFs: `Females_Codebook.pdf`, `Males_Codebook.pdf`, `Females_HOHCodebook.pdf`, `Males_HOHCodebook.pdf`
- Questionnaire PDFs: `Female_RespondentQuestionnaire.pdf`, `Male_RespondentQuestionnaire.pdf`, `HOHQuestionnaire.pdf`

**Flow:** §1 Load & auto-scan → §2 Researcher-owned `COVARIATE_MAP` → §3 Enhance from PDFs → §4 Final codebook DataFrame + TSV.

In [ ]:
from pathlib import Path
import sys

from IPython.display import display

import pandas as pd
import pyreadstat

_repo = next(
    (
        d
        for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (d / "utils" / "repo_paths.py").is_file() and (d / "data" / "raw").is_dir()
    ),
    None,
)
if _repo is None:
    raise RuntimeError("Cannot find repo root (expected utils/repo_paths.py and data/raw/).")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from utils.repo_paths import find_repo_root

ROOT = find_repo_root()
print("ROOT =", ROOT)

ZAMBIA_DIR = ROOT / "data" / "raw" / "Zambia Stata"
MALE_DTA = ZAMBIA_DIR / "ZAMBIA_VACS_2014_Male_PUD.dta"
FEMALE_DTA = ZAMBIA_DIR / "ZAMBIA_VACS_2014_Female_PUD.dta"

FEMALE_CODEBOOK = ZAMBIA_DIR / "ZAMBIA_VACS_2014_Females_Codebook.pdf"
MALE_CODEBOOK = ZAMBIA_DIR / "ZAMBIA_VACS_2014_Males_Codebook.pdf"
HOH_CODEBOOK_F = ZAMBIA_DIR / "ZAMBIA_VACS_2014_Females_HOHCodebook.pdf"
HOH_CODEBOOK_M = ZAMBIA_DIR / "ZAMBIA_VACS_2014_Males_HOHCodebook.pdf"

## 1. Load data & auto-scan

Load the male PUD as the primary frame, scan Stata labels for covariate candidates, and parse codebook PDFs for question text and response options.

In [ ]:
df, meta = pyreadstat.read_dta(MALE_DTA)
print(f"Male PUD: {df.shape[0]:,} rows × {df.shape[1]:,} cols")

df_f, meta_f = pyreadstat.read_dta(FEMALE_DTA)
print(f"Female PUD: {df_f.shape[0]:,} rows × {df_f.shape[1]:,} cols")

In [ ]:
from utils.covariates import search_dta_for_covariates

candidates = search_dta_for_covariates(df, meta)
with pd.option_context("display.max_colwidth", 80, "display.width", 220, "display.max_rows", 80):
    display(candidates)

In [ ]:
from utils.pdf_parse import (
    extract_codebook_entries,
    entries_by_question_number,
    entries_by_variable_name,
)

cb_f = extract_codebook_entries(FEMALE_CODEBOOK)
cb_m = extract_codebook_entries(MALE_CODEBOOK)
cb_hoh = extract_codebook_entries(HOH_CODEBOOK_F)

by_qnum_f = entries_by_question_number(cb_f)
by_qnum_m = entries_by_question_number(cb_m)
by_qnum_hoh = entries_by_question_number(cb_hoh)
by_var_f = entries_by_variable_name(cb_f)
by_var_m = entries_by_variable_name(cb_m)
by_var_hoh = entries_by_variable_name(cb_hoh)

print(f"Female codebook: {len(cb_f)} entries")
print(f"Male codebook:   {len(cb_m)} entries")
print(f"HOH codebook:    {len(cb_hoh)} entries")

## 2. Researcher-owned covariate mapping

Based on the auto-scan and codebook PDF review, define the mapping from covariate categories to variables. The `variable` field uses **questionnaire question numbers** (e.g. `Q2; F2` for male/female), matching the convention in the Zambia Word-doc codebook.

Helper function to look up question text and response options from parsed PDFs:

In [ ]:
def lookup(qnum_male: str, qnum_female: str = "", qnum_hoh: str = ""):
    """Look up question text and format from parsed codebook PDFs."""
    e = None
    if qnum_hoh:
        e = by_qnum_hoh.get(qnum_hoh)
    if not e and qnum_female:
        e = by_qnum_f.get(qnum_female)
    if not e and qnum_male:
        e = by_qnum_m.get(qnum_male)
    if e:
        return e.question_text, e.format_string
    return "", ""

In [ ]:
from utils.covariates import classify_variable_type

# --- Researcher-owned mapping ---
# Each entry: (category, variable_notation, qnum_male, qnum_female, qnum_hoh, type_override)
# type_override: set manually when the heuristic won't work (e.g. "Two files" for sex)

RAW_MAP = [
    # Respondent-level covariates
    ("Sex",                      "Two files",          "",      "",      "",    "categorical"),
    ("Age",                      "Q2; F2",             "M2",    "F2",    "",    ""),
    ("Highest Education Level",  "Q5; F5",             "M5",    "F5",    "",    ""),
    ("Enough Money for\u2026",   "Q7AA",               "",      "",      "",    "binary"),
    ("Enough Money for\u2026",   "Q7AB",               "",      "",      "",    "binary"),
    ("Enough Money for\u2026",   "Q7AD",               "",      "",      "",    "binary"),
    ("Lives with biological mom","Q13; F13",           "M13",   "F13",   "",    ""),
    ("Lives with biological dad","Q19; F19",           "M19",   "F19",   "",    ""),
    ("Ever Married",             "Q25; F25",           "M25",   "F25",   "",    ""),
    ("Disability",               "NA",                 "",      "",      "",    "NA"),
    ("Community Trust",          "Q36; F36",           "M36",   "F36",   "",    ""),
    ("Community Safety",         "Q37; F37",           "M37",   "F37",   "",    ""),
    ("Supportive friends",       "Q7; F7",             "M7",    "F7",    "",    ""),
    ("Engage in work for pay in last 12 months",
                                 "Q11; F11",           "M11",   "F11",   "",    ""),
    ("Drank alcohol in last 30 days",
                                 "Q1300; F1300",       "M1300", "F1300", "",    ""),
    ("Smoke Cigarettes in last 30 days",
                                 "Q1301; F1301",       "M1301", "F1301", "",    ""),
    ("Mental Health",
     "Q1303A\u2013Q1303F; F1303A\u2013F1303F",
                                                       "M1303A","F1303A","",    "categorical"),
    # Household-level covariates (HOH questionnaire)
    ("Main source of drinking water (HH)",
                                 'H4 (and H4_OT if \"other\")', "", "", "H4", ""),
    ("Flush toilet (HH)",        "H5 (and H5_OT)",    "",      "",      "H5",  ""),
    ("Shared HH",                "H6",                 "",      "",      "H6",  ""),
    ("Electricity (HH)",         "H7A\u2013H7G",       "",      "",      "H7A", ""),
    ("Dwelling floor (HH)",      "H9",                 "",      "",      "H9",  ""),
    ("Roof type (HH)",           "H10",                "",      "",      "H10", ""),
    ("Wall material (HH)",       "H11",                "",      "",      "H11", ""),
    ("# rooms in household (HH)","H12",                "",      "",      "H12", ""),
    ("# rooms in household (HH)","H13",                "",      "",      "H13", ""),
]

## 3. Enhance from PDFs

For each entry, look up question text and response options from the parsed codebook PDFs. Use `classify_variable_type()` for Type when not manually overridden.

In [ ]:
entries = []
labels_m = meta.column_names_to_labels or {}
labels_f = meta_f.column_names_to_labels or {}

for cat, var_notation, qm, qf, qh, type_ov in RAW_MAP:
    # Look up question text and format from PDFs
    q_text, q_format = lookup(qm, qf, qh)

    # Determine type
    if type_ov:
        var_type = type_ov
    elif var_notation == "NA":
        var_type = "NA"
    else:
        # Try to classify from .dta data using the Stata variable name
        # The variable name in the .dta is the codebook entry's variable_name (Q-prefixed)
        stata_var = None
        if qm:
            e = by_qnum_m.get(qm)
            if e and e.variable_name and e.variable_name in df.columns:
                stata_var = e.variable_name
        if not stata_var and qf:
            e = by_qnum_f.get(qf)
            if e and e.variable_name and e.variable_name in df_f.columns:
                stata_var = e.variable_name
        if not stata_var and qh:
            e = by_qnum_hoh.get(qh)
            if e and e.variable_name and e.variable_name in df.columns:
                stata_var = e.variable_name

        if stata_var:
            var_type = classify_variable_type(df[stata_var])
        else:
            var_type = ""

    entries.append({
        "category": cat,
        "variable": var_notation,
        "type": var_type,
        "format": q_format,
        "question": q_text,
    })

print(f"Entries built: {len(entries)}")

## 4. Harmonized covariate codebook

Build the 5-column DataFrame and produce TSV for Excel paste.

In [ ]:
from utils.covariates import build_covariate_codebook_df, covariate_codebook_to_tsv

codebook_df = build_covariate_codebook_df(entries)

with pd.option_context(
    "display.max_colwidth", 80,
    "display.width", 220,
    "display.max_rows", 40,
):
    display(codebook_df)

In [ ]:
print("\n--- TSV (copy for Excel / codebook) ---\n")
print(covariate_codebook_to_tsv(codebook_df))